In [1]:
### Data Ingestion
from langchain_core.documents import Document



In [2]:
doc=Document(
    page_content="this is main text content iam usig to create RAG",
    metadata={
       "source":"example.txt",
       "pages":1,
       "author":"Priya",
       "date_created":"2026-03-28"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Priya', 'date_created': '2026-03-28'}, page_content='this is main text content iam usig to create RAG')

In [3]:
## create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [4]:
###Text Loader
#from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader
loader=TextLoader("../data/text_files/sample_python.txt",encoding="utf-8")
document=loader.load()
print(document)



/Users/haripriyahariselvam/Desktop/RAG/RAG_PIPELINE/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/sample_python.txt'}, page_content='Python is an awesome language for coding\nPython does everything on the fly\n')]


In [6]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader=DirectoryLoader("../data/text_files",glob="**/*.txt",loader_cls=TextLoader,loader_kwargs={'encoding':'utf-8'},
                           show_progress=False)
documents=dir_loader.load()
documents

[Document(metadata={'source': '../data/text_files/sample_python.txt'}, page_content='Python is an awesome language for coding\nPython does everything on the fly\n')]

In [ ]:
##embeddings and vectorStoreDB


In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict,Any ,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
#Embedding model form hugging face
class Embeddingmanager:
    """Handle document embedding generation using SentenceTransformer"""
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """Initiallize the embedding manager
        Args:
        model_name: Hugging Face model name for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model{self.model_name}:{e}")
            raise
    def generate_embeddings(self,texts:List[str])->np.ndarray:
        """Generate embeddings for a list of texts
        Args:texts :List of text strings to embed
        Returns:
        numpy array of embeddings with shape (len(texts),embedding_dim)"""
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts..")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape:{embeddings.shape}")
        return embeddings

In [9]:
embedding_manager=Embeddingmanager()
embedding_manager

Loading embedding model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6310.91it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension:384


###vector store


In [11]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1


In [12]:
 #Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs
chunks = split_documents(documents)
print(f"Total chunks created: {len(chunks)}")


Split 1 documents into 1 chunks

Example chunk:
Content: Python is an awesome language for coding
Python does everything on the fly...
Metadata: {'source': '../data/text_files/sample_python.txt'}
Total chunks created: 1


In [13]:
###convert the text to embeddings

texts=[doc.page_content for doc in chunks]
texts

['Python is an awesome language for coding\nPython does everything on the fly']

In [14]:
#generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts)
#store in the vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.14it/s]

Generated embeddings with shape:(1, 384)
Adding 1 documents to vector store...
Successfully added 1 documents to vector store
Total documents in collection: 2


In [16]:
#RAG Retriever 

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: Embeddingmanager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [17]:
rag_retriever

In [18]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 53.77it/s]

Generated embeddings with shape:(1, 384)
Retrieved 0 documents (after filtering)


[]

In [19]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.09it/s]

Generated embeddings with shape:(1, 384)
Retrieved 0 documents (after filtering)


[]

In [28]:
#RAG PIPELINE -VectorDB to LLM Output Generation

In [34]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
#print(os.getenv("GROQ_API_KEY"))
groq_api_key=""


In [35]:
llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature =0.1,max_tokens=1024)

GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

In [37]:
def rag_simple(query,retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k=top_k)
    
    #combine retrived document contents
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"
    #generate the answer using Groq LLM
    prompt =f"""use the following context to answer the question concisely.
        Context:
        {context}
        Question:{query}
        Answer:"""
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [ ]:
answer=rag_simple("what is attention mechanism?",rag_retriever,llm)
print(answer)